# Road Anomaly Detection — Full Agent Pipeline

**Chain:** Dashcam frame &rarr; Fine-tuned YOLO11 detection &rarr; **Verification Agent** &rarr; **Context Agent** &rarr; **Severity Agent** &rarr; **Routing Agent** &rarr; Final Incident

Each agent is a separate, independently testable function. This notebook runs them in a fixed sequence (a multi-agent pipeline) on your real trained model's output.

**Original contributions in this pipeline** (the parts that are not just "using YOLO"):
1. **Verification Agent** — a multi-factor scoring rule that merges repeated frame-level detections of the same physical defect into one confirmed incident, filtering out one-off false positives.
2. **Severity Agent** — a weighted scoring formula combining detection confidence, evidence strength, contextual risk, and recurrence into a single actionable priority score.

Place your trained `best.pt` in `../models/best.pt` and your Stage 1 output in `../outputs/incidents/stage1_frames.json` before running.

In [1]:
!pip install -q ultralytics


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1 — Load the trained model and Stage 1 data

In [2]:
from ultralytics import YOLO
import json

with open("../outputs/incidents/stage1_frames.json") as f:
    stage1_frames = json.load(f)

model = YOLO("../models/best.pt")
print("Model classes:", model.names)
print(f"Loaded {len(stage1_frames)} frames from Stage 1")

Model classes: {0: 'D00_longitudinal_crack', 1: 'D10_transverse_crack', 2: 'D20_alligator_crack', 3: 'D40_pothole', 4: 'repair'}
Loaded 164 frames from Stage 1


## Step 2 — Run detection on every frame

Each detection carries its frame number, class, confidence, bounding box, GPS, and weather — this is the shared data contract every agent below consumes.

In [3]:
def detect_anomalies(frames, model, conf_threshold=0.4):
    all_detections = []
    for frame in frames:
        results = model.predict(frame["frame_path"], conf=conf_threshold, verbose=False)[0]
        for box in results.boxes:
            cls_id = int(box.cls[0])
            cls_name = model.names[cls_id]
            confidence = float(box.conf[0])
            all_detections.append({
                "frame": frame["frame_number"],
                "class": cls_name,
                "conf": round(confidence, 3),
                "bbox": box.xyxy[0].tolist(),
                "gps": frame["gps"],
                "weather": frame["weather"]
            })
    return all_detections

raw_detections = detect_anomalies(stage1_frames, model, conf_threshold=0.4)
print(f"Total raw detections: {len(raw_detections)}")
raw_detections[:3]

Total raw detections: 78


[{'frame': 0,
  'class': 'D00_longitudinal_crack',
  'conf': 0.476,
  'bbox': [2596.204833984375,
   949.4796752929688,
   3203.80810546875,
   1418.819580078125],
  'gps': [13.0827, 80.2707],
  'weather': {'condition': 'Clouds',
   'description': 'overcast clouds',
   'temp_c': 32.21,
   'humidity': 72}},
 {'frame': 0,
  'class': 'D00_longitudinal_crack',
  'conf': 0.456,
  'bbox': [1570.4234619140625,
   780.0858764648438,
   1774.2132568359375,
   1210.1617431640625],
  'gps': [13.0827, 80.2707],
  'weather': {'condition': 'Clouds',
   'description': 'overcast clouds',
   'temp_c': 32.21,
   'humidity': 72}},
 {'frame': 5,
  'class': 'D00_longitudinal_crack',
  'conf': 0.682,
  'bbox': [2872.069580078125,
   1120.431640625,
   3308.40185546875,
   1449.8939208984375],
  'gps': [13.082745, 80.270726],
  'weather': {'condition': 'Clouds',
   'description': 'overcast clouds',
   'temp_c': 32.21,
   'humidity': 72}}]

## Step 3 — Verification Agent (original contribution)

**Problem it solves:** a single physical defect is seen by the model in many consecutive video frames as the vehicle drives past it. Without this step, that one defect would generate many duplicate alerts.

**The rule:** consecutive raw detections are merged into one verified incident only if ALL of the following hold between them:
- same predicted class
- within `max_frame_gap` frames of each other
- within `max_gps_dist` metres of each other (haversine distance)
- confidence above `min_conf`

A merged group only becomes a confirmed incident once it has been seen in at least `min_streak` frames — this is the filter that removes one-off false positives.

In [4]:
from math import radians, sin, cos, sqrt, atan2

def gps_distance_meters(gps1, gps2):
    R = 6371000
    lat1, lon1 = radians(gps1[0]), radians(gps1[1])
    lat2, lon2 = radians(gps2[0]), radians(gps2[1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def build_incident(group):
    avg_conf = sum(d["conf"] for d in group) / len(group)
    return {
        "incident_id": f"RA-{group[0]['frame']:04d}",
        "anomaly": group[0]["class"],
        "status": "VERIFIED",
        "evidence_frames": len(group),
        "avg_confidence": round(avg_conf, 3),
        "gps": group[len(group)//2]["gps"],
        "frame_range": (group[0]["frame"], group[-1]["frame"])
    }

def verify_incidents(detections, min_conf=0.4, max_frame_gap=10, max_gps_dist=20, min_streak=2):
    detections = sorted(detections, key=lambda d: d["frame"])
    incidents = []
    current_group = []

    for det in detections:
        if det["conf"] < min_conf:
            continue
        if not current_group:
            current_group = [det]
            continue

        last = current_group[-1]
        same_class = det["class"] == last["class"]
        close_frame = (det["frame"] - last["frame"]) <= max_frame_gap
        close_gps = gps_distance_meters(det["gps"], last["gps"]) <= max_gps_dist

        if same_class and close_frame and close_gps:
            current_group.append(det)
        else:
            if len(current_group) >= min_streak:
                incidents.append(build_incident(current_group))
            current_group = [det]

    if len(current_group) >= min_streak:
        incidents.append(build_incident(current_group))

    return incidents

verified_incidents = verify_incidents(raw_detections)
print(f"Raw detections: {len(raw_detections)}  ->  Verified incidents: {len(verified_incidents)}")
for inc in verified_incidents:
    print(inc)

Raw detections: 78  ->  Verified incidents: 18
{'incident_id': 'RA-0000', 'anomaly': 'D00_longitudinal_crack', 'status': 'VERIFIED', 'evidence_frames': 5, 'avg_confidence': 0.519, 'gps': [13.082745, 80.270726], 'frame_range': (0, 10)}
{'incident_id': 'RA-0020', 'anomaly': 'D00_longitudinal_crack', 'status': 'VERIFIED', 'evidence_frames': 2, 'avg_confidence': 0.531, 'gps': [13.082879, 80.270806], 'frame_range': (20, 20)}
{'incident_id': 'RA-0035', 'anomaly': 'D00_longitudinal_crack', 'status': 'VERIFIED', 'evidence_frames': 4, 'avg_confidence': 0.727, 'gps': [13.083058, 80.270911], 'frame_range': (35, 45)}
{'incident_id': 'RA-0080', 'anomaly': 'D00_longitudinal_crack', 'status': 'VERIFIED', 'evidence_frames': 2, 'avg_confidence': 0.514, 'gps': [13.083506, 80.271175], 'frame_range': (80, 90)}
{'incident_id': 'RA-0110', 'anomaly': 'D00_longitudinal_crack', 'status': 'VERIFIED', 'evidence_frames': 2, 'avg_confidence': 0.436, 'gps': [13.08373, 80.271307], 'frame_range': (110, 115)}
{'incide

## Step 4 — Context Agent

**Problem it solves:** the same detection means different things under different conditions. Water on the road during active rain is expected and self-resolving; the same water with no rain and repeat sightings points to a drainage fault. This agent produces that interpretation, plus a rule-based `risk_level` that later agents rely on.

In [5]:
def context_agent(incident, weather, repetition_count=1):
    anomaly = incident["anomaly"]
    condition = weather.get("condition", "Unknown")
    is_rainy = condition in ("Rain", "Drizzle", "Thunderstorm")

    context = {
        "incident_id": incident["incident_id"],
        "anomaly": anomaly,
        "weather_condition": condition,
        "repetition_count": repetition_count,
        "conclusion": None,
        "risk_level": "Normal"
    }

    if "water" in anomaly.lower():
        if is_rainy:
            context["conclusion"] = "Water accumulation likely related to current rainfall"
        elif repetition_count > 1:
            context["conclusion"] = "Possible persistent drainage issue"
            context["risk_level"] = "Increased"
        else:
            context["conclusion"] = "Water presence unexplained by weather — needs inspection"
            context["risk_level"] = "Increased"
    elif "crack" in anomaly.lower():
        if is_rainy:
            context["conclusion"] = "Crack may worsen faster due to water seepage"
            context["risk_level"] = "Increased"
        elif repetition_count > 1:
            context["conclusion"] = "Recurring crack — monitor for expansion"
            context["risk_level"] = "Increased"
        else:
            context["conclusion"] = "Standard crack, no weather-related urgency"
    elif "pothole" in anomaly.lower():
        if is_rainy:
            context["conclusion"] = "Pothole with active rain — increased vehicle hazard"
            context["risk_level"] = "Increased"
        elif repetition_count > 1:
            context["conclusion"] = "Recurring pothole — likely worsening, needs prioritized repair"
            context["risk_level"] = "Increased"
        else:
            context["conclusion"] = "Pothole detected, standard road hazard"
    else:
        context["conclusion"] = "General road anomaly detected"

    return context

print("Context Agent ready.")

Context Agent ready.


### Optional — Ollama hybrid explanation layer

The Context Agent above makes the actual decision (`conclusion`, `risk_level`). This function only asks a local LLM to phrase that already-decided result in natural language for a dashboard — the LLM never decides anything itself, it only explains. Requires Ollama running locally with `llama3.2:3b` pulled. Skip this cell if Ollama isn't set up; the pipeline works fully without it.

In [6]:
try:
    import ollama
    OLLAMA_AVAILABLE = True
except ImportError:
    OLLAMA_AVAILABLE = False
    print("ollama package not installed — skipping LLM explanation layer (pipeline still works without it).")

def context_agent_llm(incident, weather, repetition_count=1):
    rule_based_result = context_agent(incident, weather, repetition_count)
    if not OLLAMA_AVAILABLE:
        rule_based_result["llm_explanation"] = None
        return rule_based_result

    prompt = f"""You are a road maintenance assistant. Given this structured incident data, write a single clear, professional sentence explaining the situation for a government dashboard. Do not add information not given below.

Incident ID: {rule_based_result['incident_id']}
Anomaly type: {rule_based_result['anomaly']}
Weather condition: {rule_based_result['weather_condition']}
Times detected: {rule_based_result['repetition_count']}
Preliminary conclusion: {rule_based_result['conclusion']}
Risk level: {rule_based_result['risk_level']}

Write only the explanation sentence, nothing else."""

    try:
        response = ollama.chat(model="llama3.2:3b", messages=[{"role": "user", "content": prompt}])
        rule_based_result["llm_explanation"] = response["message"]["content"].strip()
    except Exception as e:
        rule_based_result["llm_explanation"] = None
        print("Ollama call failed, continuing without LLM explanation:", e)

    return rule_based_result

print("Context Agent (LLM-hybrid) ready.")

Context Agent (LLM-hybrid) ready.


## Step 5 — Severity Agent (original contribution)

**Problem it solves:** a department receiving many alerts per day cannot inspect all of them immediately. This agent turns the incident's evidence, the Context Agent's risk judgment, and how often the same defect has been seen into one priority score.

**The formula:**

```
severity_score =
      avg_confidence * 30                          (detection certainty)
    + min(evidence_frames, 10) * 2                 (strength of visual evidence)
    + risk_weight[context.risk_level]               (0 for Normal, 25 for Increased)
    + min(repetition_count, 5) * 8                  (recurrence over time)
```

Thresholds: `>=70` CRITICAL, `>=50` HIGH, `>=30` MEDIUM, else LOW. The caps (`min(x, 10)`, `min(x, 5)`) prevent one factor from dominating the score indefinitely.

In [7]:
def severity_agent(incident, context_result, repetition_count=1):
    score = 0
    score += incident["avg_confidence"] * 30
    score += min(incident["evidence_frames"], 10) * 2

    risk_weights = {"Normal": 0, "Increased": 25}
    score += risk_weights.get(context_result["risk_level"], 0)

    score += min(repetition_count, 5) * 8

    if score >= 70:
        severity = "CRITICAL"
    elif score >= 50:
        severity = "HIGH"
    elif score >= 30:
        severity = "MEDIUM"
    else:
        severity = "LOW"

    return {"incident_id": incident["incident_id"], "severity_score": round(score, 1), "severity": severity}

print("Severity Agent ready.")

Severity Agent ready.


## Step 6 — Routing Agent

**Problem it solves:** different anomaly types belong to different government departments, and someone still has to decide how urgently to act. This agent maps anomaly &rarr; department and severity &rarr; recommended action.

In [8]:
DEPARTMENT_MAP = {
    "D00_longitudinal_crack": "Road Maintenance",
    "D10_transverse_crack": "Road Maintenance",
    "D20_alligator_crack": "Road Maintenance",
    "D40_pothole": "Road Maintenance",
    "repair": None,   # marks an already-repaired section — no action needed
}

def routing_agent(incident, severity_result):
    anomaly = incident["anomaly"]
    department = DEPARTMENT_MAP.get(anomaly, "General Public Works")

    if department is None:
        return {
            "incident_id": incident["incident_id"],
            "status": "IGNORED",
            "reason": "Marked as already-repaired section, no action needed"
        }

    if severity_result["severity"] == "CRITICAL":
        action = "Immediate inspection and repair required within 24 hours"
    elif severity_result["severity"] == "HIGH":
        action = "Priority inspection within 3-5 days"
    elif severity_result["severity"] == "MEDIUM":
        action = "Schedule routine inspection within 2 weeks"
    else:
        action = "Log for periodic maintenance review"

    return {
        "incident_id": incident["incident_id"],
        "department": department,
        "severity": severity_result["severity"],
        "recommended_action": action
    }

print("Routing Agent ready.")

Routing Agent ready.


## Step 7 — Final Incident assembly

Combines every agent's output into the single record your dashboard will display.

In [9]:
def build_final_incident(incident, context_result, severity_result, routing_result):
    return {
        "incident_id": incident["incident_id"],
        "anomaly": incident["anomaly"],
        "status": routing_result.get("status", "VERIFIED"),
        "gps": incident["gps"],
        "weather": context_result["weather_condition"],
        "evidence_frames": incident["evidence_frames"],
        "avg_confidence": incident["avg_confidence"],
        "context_conclusion": context_result["conclusion"],
        "llm_explanation": context_result.get("llm_explanation"),
        "risk_level": context_result["risk_level"],
        "severity": severity_result["severity"],
        "severity_score": severity_result["severity_score"],
        "department": routing_result.get("department", "N/A"),
        "recommended_action": routing_result.get("recommended_action", routing_result.get("reason", "N/A"))
    }

print("Final Incident assembler ready.")

Final Incident assembler ready.


## Step 8 — Run the full chain on every verified incident

This is the complete pipeline: real detections from your trained model, verified, contextualized against real weather, scored for severity, and routed to a department.

In [10]:
final_incidents = []

for inc in verified_incidents:
    # look up the real weather recorded for the frame this incident started at
    matching_frame = next((f for f in stage1_frames if f["frame_number"] == inc["frame_range"][0]), None)
    weather_data = matching_frame["weather"] if matching_frame else {"condition": "Unknown"}

    context_result = context_agent_llm(inc, weather_data, repetition_count=1)
    severity_result = severity_agent(inc, context_result, repetition_count=1)
    routing_result = routing_agent(inc, severity_result)
    final = build_final_incident(inc, context_result, severity_result, routing_result)
    final_incidents.append(final)

for f in final_incidents:
    print(f)
    print("---")

{'incident_id': 'RA-0000', 'anomaly': 'D00_longitudinal_crack', 'status': 'VERIFIED', 'gps': [13.082745, 80.270726], 'weather': 'Clouds', 'evidence_frames': 5, 'avg_confidence': 0.519, 'context_conclusion': 'Standard crack, no weather-related urgency', 'llm_explanation': 'A longitudinal crack has been detected on the road surface, classified as a standard crack with no urgent weather-related consequences.', 'risk_level': 'Normal', 'severity': 'MEDIUM', 'severity_score': 33.6, 'department': 'Road Maintenance', 'recommended_action': 'Schedule routine inspection within 2 weeks'}
---
{'incident_id': 'RA-0020', 'anomaly': 'D00_longitudinal_crack', 'status': 'VERIFIED', 'gps': [13.082879, 80.270806], 'weather': 'Clouds', 'evidence_frames': 2, 'avg_confidence': 0.531, 'context_conclusion': 'Standard crack, no weather-related urgency', 'llm_explanation': 'A longitudinal crack has been detected on the road surface, classified as a standard crack with no weather-related urgency, under normal wea

## Step 9 — Save the results

This becomes the data source for the dashboard.

In [11]:
import json

with open("../outputs/incidents/final_incidents.json", "w") as f:
    json.dump(final_incidents, f, indent=2)

print(f"Saved {len(final_incidents)} final incidents to final_incidents.json")

Saved 18 final incidents to final_incidents.json


## Notes for the report

- **Fine-tuned YOLO11s**: mAP50 0.506 overall (79 epochs); D40_pothole 0.655 mAP50 (best-performing class); `repair` class weakest at 0.393 mAP50, a known limitation from class imbalance.
- **Verification Agent** and **Severity Agent** are the two original algorithmic contributions in this pipeline — both are deterministic, testable in isolation, and explainable (every score traces back to named factors).
- **Context Agent's LLM layer** is a hybrid design: the LLM only paraphrases an already-made decision into natural language, never makes the decision itself — this keeps the safety-relevant logic deterministic while still giving genuine use to a local LLM.
- **Current scope is a fixed, sequential multi-agent pipeline**, not autonomous "agentic AI" — an LLM orchestrator that decides which agents to call via tool-calling is scoped as future work, not yet implemented here.